In [ ]:
# Run from the repository root so data/, scripts/, dashboard/, outputs/ paths resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Supplementary Analyses: Reproduction Notebook

Single runnable notebook reproducing the analyses referenced in the manuscript and the
Supplementary Materials. **Part A** holds self-contained computations for the supplement
tables that had no standalone script. **Part B** runs the existing analysis scripts in
place (each remains the single source of truth) and captures their console output.

**Environment.** Kernel *Python 3 (EyeTrack)* = `/usr/local/bin/python3`
(pandas, numpy<2, scipy, scikit-learn, xgboost, statsmodels, shap).
**Scope.** Sign + Animal probes (*n* = 439, 37 operators) unless a script defines its own.

| Section | Reproduces | Manuscript / supplement item |
|---|---|---|
| A1 | Outcome-structure PCA + within/between coupling | §3 shared error-and-uncertainty axis |
| A2 | Predictor collinearity (VIF) | Supplementary Table |
| A3 | Pre-query predictor distributions | Supplementary Table |
| A4 | Dwell and failure rate by query type | Supplementary Table |
| B1 | Feature-set parsimony (full vs reduced) | Table 3 |
| B2 | Feature-block / model-class comparison | §3.5, model-class robustness |
| B3 | Confidence detector | §3.4, Supplementary Table |
| B4 | SA-state indicator failure rates by band | §3.11 |
| B5 | Bootstrap CI stability | Supplementary Table |
| B6 | Overconfidence and operator triage | §4.7 |
| B7 | Collisions, looking-vs-seeing, stability buffer | §3.6, Supplementary Figure, Supplementary Note |
| B8 | GLMM outlier sensitivity | §3.7 robustness |
| B9 | Behavioural correlation matrix | supporting correlations |
| B10 | Permutation testing (no-skill and within-operator nulls) | Supplementary Note (slow) |
| B11 | Saturating dwell-dependence (SHAP) | Supplementary Figure (slow) |

## Setup

In [1]:
import os, sys, subprocess, textwrap
import pandas as pd, numpy as np

PROJ = os.getcwd()   # this notebook's own folder; scripts and data live alongside it
PYEXE = sys.executable            # kernel interpreter = /usr/local/bin/python3
ENV = dict(os.environ, MPLBACKEND='Agg')

def run_analysis(script, timeout=900):
    """Run an analysis script in a fresh process and echo its console output."""
    print(f"$ python {script}")
    print("=" * 78)
    try:
        r = subprocess.run([PYEXE, script], capture_output=True, text=True,
                           timeout=timeout, env=ENV, cwd=PROJ)
    except subprocess.TimeoutExpired:
        print(f"[timed out after {timeout}s, run manually if needed]")
        return
    print(r.stdout, end="")
    if r.returncode != 0:
        print("\n--- SECTION FAILED (stderr tail) ---")
        print("\n".join(r.stderr.strip().splitlines()[-8:]))

def load_scope():
    """Sign+Animal trial-level frame with the six GLMM predictors coerced."""
    df = pd.read_csv('data/master_routes_1_to_6_latency.csv')
    d = df[df['Question_Type'].isin(['Sign', 'Animal'])].copy()
    return d

PRED = ['Before_Dwell_Proportion_Target_Object', 'Before_Speed_Variance',
        'Before_Major_SRR', 'Before_TRR', 'Before_Saccade_Rate_Hz',
        'Before_Road_Gaze_Pct']
print("cwd:", os.getcwd())
print("interpreter:", PYEXE)

cwd: /Users/Ryan/EyeTrack
interpreter: /usr/local/bin/python3


# Part A: Inline supplement reproductions

Self-contained; these back supplement tables.

## A1. Outcome-structure PCA and within/between-operator coupling
Backs the §3 "shared error-and-uncertainty axis" paragraph and the new Methods sentence in
§2.12. (Full standalone version: `Outcome_Structure_PCA.ipynb`.)

In [2]:
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

d = load_scope()
for c in ['Target_Accuracy', 'Target_Latency', 'Target_Confidence']:
    d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=['Target_Accuracy', 'Target_Latency', 'Target_Confidence', 'Participant_ID'])
acc, lat, conf = d['Target_Accuracy'], d['Target_Latency'], d['Target_Confidence']

print(f"n={len(d)} probes, {d['Participant_ID'].nunique()} operators")
print(f"Spearman  latency~confidence {spearmanr(lat,conf).correlation:+.3f}"
      f" | latency~accuracy {spearmanr(lat,acc).correlation:+.3f}")

X = StandardScaler().fit_transform(d[['Target_Accuracy','Target_Latency','Target_Confidence']])
pca = PCA().fit(X)
print(f"PC1 variance explained {100*pca.explained_variance_ratio_[0]:.1f}%")
print("PC1 loadings [acc, lat, conf]:", np.round(pca.components_[0], 3))

def within_between(sub, a, b):
    g = sub.groupby('Participant_ID')
    between = spearmanr(g[a].mean(), g[b].mean()).correlation
    res = sub[[a, b]] - g[[a, b]].transform('mean')
    within = spearmanr(res[a], res[b]).correlation
    return within, between
for a, b, lab in [('Target_Latency','Target_Accuracy','latency~accuracy'),
                  ('Target_Latency','Target_Confidence','latency~confidence')]:
    w, bt = within_between(d, a, b)
    print(f"{lab:20s} within {w:+.3f} | between {bt:+.3f}")

n=439 probes, 37 operators
Spearman  latency~confidence -0.498 | latency~accuracy -0.313
PC1 variance explained 61.3%
PC1 loadings [acc, lat, conf]: [ 0.586 -0.503  0.636]
latency~accuracy     within -0.227 | between -0.484
latency~confidence   within -0.501 | between -0.316


## A2. Predictor collinearity: naive VIF (Supplementary Table)

In [3]:
from numpy.linalg import inv
d = load_scope()
for c in PRED: d[c] = pd.to_numeric(d[c], errors='coerce')
X = d.dropna(subset=PRED)[PRED].astype(float)
Xs = (X - X.mean()) / X.std()
vif = np.diag(inv(np.corrcoef(Xs.values, rowvar=False)))
print("Naive variance inflation factors (Sign+Animal):")
for c, v in zip(PRED, vif):
    print(f"  {c:40s} {v:5.2f}")
print(f"\nmax VIF = {vif.max():.2f}  (all well below the conventional 5/10 thresholds)")

Naive variance inflation factors (Sign+Animal):
  Before_Dwell_Proportion_Target_Object     1.71
  Before_Speed_Variance                     1.42
  Before_Major_SRR                          1.06
  Before_TRR                                1.37
  Before_Saccade_Rate_Hz                    1.13
  Before_Road_Gaze_Pct                      1.53

max VIF = 1.71  (all well below the conventional 5/10 thresholds)


## A3. Pre-query behavioural predictor distributions (Supplementary Table)

In [4]:
d = load_scope()
for c in PRED: d[c] = pd.to_numeric(d[c], errors='coerce')
dist = d[PRED].agg(['mean', 'std', 'median', 'min', 'max']).T
dist.columns = ['M', 'SD', 'Median', 'Min', 'Max']
print(dist.round(3).to_string())

                                           M     SD  Median    Min    Max
Before_Dwell_Proportion_Target_Object  0.063  0.103   0.008  0.000  0.664
Before_Speed_Variance                  0.306  0.364   0.166  0.000  2.435
Before_Major_SRR                       0.139  0.126   0.200  0.000  0.600
Before_TRR                             0.175  0.202   0.000  0.000  0.800
Before_Saccade_Rate_Hz                 1.810  0.648   1.800  0.000  4.200
Before_Road_Gaze_Pct                   0.767  0.165   0.804  0.159  1.000


## A4. Pre-query target-object dwell and failure rate by query type (Supplementary Table)

In [5]:
d = load_scope()
d['Target_Accuracy'] = pd.to_numeric(d['Target_Accuracy'], errors='coerce')
d['fail'] = 1 - d['Target_Accuracy']
d['dwell'] = pd.to_numeric(d['Before_Dwell_Proportion_Target_Object'], errors='coerce')
g = d.groupby('Question_Type').agg(n=('fail', 'size'),
                                   dwell_mean=('dwell', 'mean'),
                                   dwell_median=('dwell', 'median'),
                                   fail_rate=('fail', 'mean'))
print(g.round(3).to_string())

                 n  dwell_mean  dwell_median  fail_rate
Question_Type                                          
Animal         218       0.058         0.015      0.206
Sign           221       0.068         0.007      0.226


# Part B: Script-backed analyses

Each cell runs the existing analysis script unchanged and echoes its output.

## B1. Feature-set parsimony: full vs reduced (Table 3)

In [6]:
run_analysis('scripts/_parsimony_sa.py', timeout=900)

$ python _parsimony_sa.py


Sign+Animal n=439 | FULL=16 feats, REDUCED=11 feats, DISTRACT=['Before_Dwell_Proportion_speed_Distractor', 'Before_Dwell_Proportion_name_Distractor', 'Before_Dwell_Proportion_map_Distractor']

detector       base     FULL  REDUCED             verdict
Accuracy      0.216    0.307 (1.42)    0.300 (1.39)   full edge
Freeze        0.137    0.257 (1.88)    0.266 (1.94)   reduced wins
Confidence    0.298    0.382 (1.28)    0.365 (1.22)   full edge


## B2. Feature-block / model-class comparison (§3.5)

In [7]:
run_analysis('scripts/_feature_block_comparison.py', timeout=900)

$ python _feature_block_comparison.py



TARGET: Accuracy (incorrect SA)    n=659  positives=104 (15.8%)  participants=37
Block                   k   PR-AUC          95% CI   lift    ROC
------------------------------------------------------------------------------
Gaze/attention          5    0.165  [0.124,0.222]   1.04  0.532
Manual-control          6    0.188  [0.141,0.256]   1.19  0.569
Individual-diffs        2    0.154  [0.116,0.209]   0.98  0.504
Distractor-dwell        3    0.122  [0.089,0.167]   0.77  0.384
Gaze + Control         11    0.192  [0.144,0.248]   1.21  0.565
Parsimonious (3)        3    0.202  [0.144,0.265]   1.28  0.574
Target-dwell only(1)    1    0.167  [0.135,0.206]   1.06  0.546
FULL (current)         16    0.180  [0.134,0.239]   1.14  0.544

TARGET: Latency (freeze >=3.5s)    n=659  positives=71 (10.8%)  participants=37
Block                   k   PR-AUC          95% CI   lift    ROC
------------------------------------------------------------------------------
Gaze/attention          5    0.194  [

## B3. Confidence detector (§3.4, Supplementary Table)

In [8]:
run_analysis('scripts/_conf_sa.py', timeout=600)

$ python _conf_sa.py


CONFIDENCE detector, Sign+Animal (n=439, pos=131, base=0.298):
  PR-AUC 0.365  95% CI [0.282, 0.473]   (vs chance 0.298, 1.22-fold lift)
  ROC-AUC 0.604


## B4. SA-state indicator failure rates by band (§3.11)

Canonical pipeline: `attn_risk = max(error_dial, latency_dial)` (worst of the two objective dials, no dilution), smoothed into a trailing EWMA recent-state, then banded at ~85%/50% recall with a hard hazard→Red rule. This is what the deployed dashboard (`sa_dashboard_app_v3.py`) reads and what Figure 9 depicts.

In [2]:
# Regenerate the calibrated dials + composite, then band the smoothed recent-state.
run_analysis('dashboard/_build_dashboard_data.py', timeout=900)

import pandas as pd, numpy as np
from sklearn.metrics import precision_recall_curve
D = pd.read_csv('dashboard/SA_Dashboard_Data.csv')
y, lap, s = D['any_fail'].values, D['lapse'].values, D['attn_risk_recent'].values
prec, rec, thr = precision_recall_curve(y, s)
amber = float(thr[np.argmin(np.abs(rec[:-1] - 0.85))])
red   = float(thr[np.argmin(np.abs(rec[:-1] - 0.50))])
amber = min(amber, red)
state = np.where((s >= red) | (lap == 1), 'Red', np.where(s >= amber, 'Amber', 'Green'))
t = pd.DataFrame({'state': state, 'fail': y})
band = t.groupby('state')['fail'].agg(['size', 'mean']).reindex(['Green', 'Amber', 'Red'])
f = t[t.fail == 1]
print('\nSA-state failure rates (smoothed recent-state band; backs Section 3.11):')
print(band.rename(columns={'size': 'n', 'mean': 'fail_rate'}).round(3).to_string())
print(f"Amber+ recall {(f.state!='Green').mean():.0%} | Red-share {(f.state=='Red').mean():.0%} | "
      f"false-caution {(t[t.fail==0].state!='Green').mean():.0%}")

$ python _build_dashboard_data.py


WROTE SA_Dashboard_Data.csv (439 probes), SA_Dashboard_Ops.csv (37 ops), SA_Dashboard_Meta.csv
Amber>=0.183 Red>=0.285 | base error=0.216 latency=0.137
dominant mode (lift-corrected): error=23 latency=14
Spearman(triage,fail_rate)=0.58 p=0.000 | dial ROC err/lat/uns = 0.66/0.69/0.60
CONFIDENCE CHECK (Red-triggered): prompts=145 (33%) | confirmed=63 (37/63 errors, 59%) | overconfident=66 (9 actual errors) | inject level=0.59
Top 5 triage: [{'rank': 1, 'Participant_ID': '01_RT', 'attn': 0.36822210086716545, 'dominant_mode': 'error', 'fails': 6}, {'rank': 2, 'Participant_ID': '36_RT', 'attn': 0.3450106903910637, 'dominant_mode': 'latency', 'fails': 10}, {'rank': 3, 'Participant_ID': '33_RT', 'attn': 0.34325454086065293, 'dominant_mode': 'latency', 'fails': 4}, {'rank': 4, 'Participant_ID': '08_RT', 'attn': 0.3372141669193904, 'dominant_mode': 'error', 'fails': 2}, {'rank': 5, 'Participant_ID': '11_RT', 'attn': 0.32289766714916573, 'dominant_mode': 'error', 'fails': 4}]
20_RT rank: 32



SA-state failure rates (smoothed recent-state band; backs Section 3.11):
         n  fail_rate
state                
Green  148      0.108
Amber  134      0.321
Red    157      0.452
Amber+ recall 88% | Red-share 55% | false-caution 57%


## B5. Bootstrap confidence-interval stability (Supplementary Table)

## B6. Overconfidence and operator triage (§4.7)

In [11]:
run_analysis('scripts/_overconfidence_risk.py', timeout=300)

$ python _overconfidence_risk.py


OPERATOR-LEVEL (n=37): overconfident-error rate vs risky driving
  overconf_rate vs collisions_rate : rho=+0.22 p=0.186  | partial(ctrl error_rate) rho=+0.10 p=0.559
  overconf_rate vs speeding_rate   : rho=-0.19 p=0.263  | partial(ctrl error_rate) rho=-0.35 p=0.037
  overconf_rate vs hazard_rate     : rho=+0.01 p=0.934  | partial(ctrl error_rate) rho=-0.03 p=0.841
  overconf_rate vs mean_speed      : rho=-0.12 p=0.463  | partial(ctrl error_rate) rho=-0.25 p=0.149

  (reference) overall error_rate vs collisions_rate: rho=0.36 p=0.030

CONTINUOUS overconfidence index = mean confidence on incorrect trials (n operators with >=1 error):
  available for 33/37 operators
  conf_when_wrong vs collisions_rate : rho=-0.16 p=0.369
  conf_when_wrong vs speeding_rate   : rho=-0.11 p=0.539
  conf_when_wrong vs hazard_rate     : rho=+0.03 p=0.854
  conf_when_wrong vs mean_speed      : rho=-0.38 p=0.028

TRIAL-LEVEL (n=659): does an overconfident error co-occur with risk on the SAME query?
  collision

## B7. Collisions, looking-vs-seeing, stability buffer (§3.6, Supplementary Figure, Supplementary Note)

In [ ]:
run_analysis('scripts/_collision_stability_figure.py', timeout=300)

## B8. GLMM outlier sensitivity (§3.7 robustness)

In [13]:
run_analysis('scripts/_glmm_outlier_sensitivity.py', timeout=600)

$ python _glmm_outlier_sensitivity.py


OUTCOME: INCORRECT   (Sign+Animal, n=439)
predictor                RAW OR [95% CI]        WINSORIZED OR [95% CI]
----------------------------------------------------------------------------
z_tgt_dwell     0.64 [0.45,0.91] *          0.63 [0.44,0.88] *
z_speed_var     1.29 [1.02,1.64] *          1.32 [1.04,1.67] *
z_major_srr     1.34 [1.04,1.72] *          1.31 [1.02,1.68] *
z_trr           1.02 [0.79,1.31]            1.02 [0.79,1.31]  
z_saccade       0.92 [0.71,1.19]            0.92 [0.71,1.19]  
z_road_gaze     1.13 [0.86,1.50]            1.12 [0.85,1.48]  

OUTCOME: UNSURE   (Sign+Animal, n=439)
predictor                RAW OR [95% CI]        WINSORIZED OR [95% CI]
----------------------------------------------------------------------------
z_tgt_dwell     0.54 [0.39,0.73] *          0.55 [0.41,0.75] *
z_speed_var     1.04 [0.84,1.30]            1.05 [0.85,1.31]  
z_major_srr     1.08 [0.86,1.35]            1.06 [0.85,1.33]  
z_trr           1.30 [1.04,1.62] *          1.31 [1.05,

## B9. Behavioural correlation matrix

In [2]:
run_analysis('scripts/Analyse_Correlations.py', timeout=300)

$ python Analyse_Correlations.py


📂 Loading master dataset: master_routes_1_to_6_combined.csv...
⚖️ Processing dwell proportions and inverting accuracy for failure tracking...

📊 PEARSON CORRELATION MATRIX (r-values)
                             Response Latency (s)  Cognitive Failure (Err)  Gaze Dwell %: Target Object  Vehicle Speed Variance  Steering Variance  Mean Saccadic Velocity  Road Gaze Pct (%)  Scanpath Rate (px/s)
Response Latency (s)                        1.000                    0.278                       -0.275                   0.173              0.019                  -0.024              0.173                -0.034
Cognitive Failure (Err)                     0.278                    1.000                       -0.141                   0.143              0.020                  -0.029              0.110                -0.027
Gaze Dwell %: Target Object                -0.275                   -0.141                        1.000                  -0.210             -0.092                   0.029           

## B10. Permutation testing: no-skill and within-operator nulls (Supplementary Note): slow

In [15]:
run_analysis('scripts/_perm_vs_dummy.py', timeout=900)

$ python _perm_vs_dummy.py


Permutation test vs no-skill null | Sign+Animal n=439 | reduced-11 feats | B=1000

== Accuracy (incorrect) | learner=xgb | base rate(=PR-AUC null)=0.216 ==
   PR-AUC observed = 0.300  (lift 1.39x over base)
     free-perm null mean=0.221 (95% 0.178-0.282)  p=0.0100
     within-op null mean=0.241 (95% 0.195-0.302)  p=0.0300
   ROC-AUC observed = 0.660  (null=0.500)
     free-perm  p=0.0010   within-op p=0.0020
   [346s]

== Latency (delayed >=3.5s) | learner=rf | base rate(=PR-AUC null)=0.137 ==
   PR-AUC observed = 0.266  (lift 1.94x over base)
     free-perm null mean=0.144 (95% 0.108-0.196)  p=0.0010
     within-op null mean=0.182 (95% 0.135-0.244)  p=0.0110
   ROC-AUC observed = 0.689  (null=0.500)
     free-perm  p=0.0010   within-op p=0.0060
   [480s]

== Confidence (unsure <=4) | learner=lr | base rate(=PR-AUC null)=0.298 ==
   PR-AUC observed = 0.365  (lift 1.22x over base)
     free-perm null mean=0.304 (95% 0.255-0.367)  p=0.0300
     within-op null mean=0.340 (95% 0.286-0.400

## B11. Saturating dwell-dependence, SHAP (Supplementary Figure): slow

In [16]:
run_analysis('scripts/_dwell_dependence_combined.py', timeout=900)

$ python _dwell_dependence_combined.py


saved Supp_TargetDwell_Dependence.png (two-panel: accuracy + latency)


## B12. Nested leave-operators-out transfer check (§3.11 out-of-fold figures)

Fully nested validation behind the §3.11 sentence *"...the Red-state failure rate reading about 39% out of fold against about 44% in sample..."*. Outer `GroupKFold`-5 over operators; within each fold the dials, the isotonic calibration (inner `GroupKFold`-4 OOF), and both cut-points are fitted on the training operators only and evaluated on the held-out ones. `T1/T2` gives the in-sample vs nested-OOF Red-state failure rate (≈0.44 vs ≈0.39, base ≈0.30) and the per-fold held-out recall scatter (Amber stable, Red wide); `T3` shows alert-budget / cost-weighted operating points transferring; `T4` is an underpowered per-operator-offset demo. Exploratory support only, not a headline result.

In [2]:
run_analysis('scripts/deployment_validation_v3.py', timeout=600)

$ python deployment_validation_v3.py


n=439  ops=37  base any_fail=0.296

===== T1/T2  recall + state-conditional failure rates =====
IN-SAMPLE : Amber recall=0.854 rate=0.656 | Red recall=0.500 prec=0.439 rate=0.337
           P(fail): Green=0.126 Amber=0.329 Red=0.439   (base 0.296)
NESTED-OOF: Amber recall=0.900 rate=0.731 | Red recall=0.485 prec=0.394 rate=0.364
           P(fail): Green=0.110 Amber=0.335 Red=0.394   (base 0.296)
per-fold cut-points: Amber 0.167-0.185  Red 0.266-0.286
per-fold held-out recall SD: Amber 0.035 (range 0.85-0.95)  Red 0.152 (range 0.29-0.70)

===== T2b  composite -> P(any_fail) calibration reliability =====
Brier: in-sample=0.1817  nested-OOF=0.1907
nested reliability (5 bins):
                     pred       obs    n
q                                       
(-0.001, 0.134]  0.117565  0.106796  103
(0.134, 0.232]   0.163787  0.197368   76
(0.232, 0.402]   0.366641  0.386555  119
(0.402, 0.479]   0.445359  0.402299   87
(0.479, 0.667]   0.506911  0.425926   54

===== T3  operating-point sel

## B13. Cross-validation robustness to operator sex and age (Supplementary Note)

Tests whether the grouped cross-validation was biased by the uneven distribution of operator sex and age across folds. `GroupKFold` partitions operators but does not balance demographics, and both sex and age show between-operator associations with the outcomes (§3.7). Reports (1) the demographic composition of the five test folds, (2) out-of-fold PR-AUC with folds stratified by sex and by age band (`StratifiedGroupKFold`, operators intact), and (3) the effect of adding sex and age as predictors. All three move PR-AUC by <0.03 per outcome, so the behavioural signal is not a demographic proxy. Complements the GLMM age/sex covariate check (§3.7) and the within-operator permutation nulls (Supplementary Note).

In [2]:
run_analysis('scripts/_demographic_cv_sensitivity.py', timeout=600)

$ python _demographic_cv_sensitivity.py


n = 439 probes, 37 operators (16 F / 21 M; 13 in older bands 4-6). Age_Num is an ordinal band (1-6).

(1) GroupKFold test-fold composition (as used in the paper):
 fold  ops  F  M  female_frac  older_frac  err_rate  lat_rate
    1    8  2  6         0.25        0.50      0.17      0.18
    2    7  4  3         0.57        0.00      0.26      0.13
    3    7  3  4         0.43        0.43      0.23      0.08
    4    7  4  3         0.57        0.29      0.15      0.14
    5    8  3  5         0.38        0.50      0.27      0.14
    female fraction spans 25%-57%; older fraction spans 0%-50%; error rate spans 0.15-0.27

(2) Out-of-fold PR-AUC (ROC-AUC) by cross-validation scheme:
outcome                     base    GroupKFold     Strat-sex     Strat-age
Accuracy (XGBoost)         0.216  0.300(0.660)  0.306(0.651)  0.329(0.667)
Latency (random forest)    0.137  0.266(0.689)  0.248(0.684)  0.238(0.688)
Low-confidence (logistic)  0.298  0.365(0.604)  0.344(0.583)  0.339(0.586)

(3) Adding 